In [7]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery DataFrames - Local Peek Cache (Experimental)

This notebook introduces and demonstrates the experimental **local peek cache** feature in BigQuery DataFrames (`bigframes`).

## Overview

When doing interactive data science in Jupyter notebooks, a common workflow is to perform a transformation, look at the first few rows (using `.head()`, `.peek()`, or by simply displaying the DataFrame), and then perform another transformation.

By default, BigQuery DataFrames translates each of these steps into a BigQuery query. Even if you are only looking at the first 10 rows, BigQuery might scan gigabytes or terabytes of data to compute the results. This results in:
1. **High query latency** (each cell execution takes a few seconds).
2. **High query costs** (each query incurs scanning costs).

The **Local Peek Cache** addresses this. When enabled:
* The first time you peek at a DataFrame, BigQuery DataFrames pulls a larger local sample (e.g. 10,000 rows) and caches it in memory.
* Subsequent compatible operations (e.g. column projections, filters, math operations, custom UDFs) on that DataFrame that also request a peek will be computed **locally** (via Polars) on the cached sample.
* This completely avoids BigQuery execution for intermediate checks, making your interactive loop feel **instantaneous** and **cost-free**.

---

### In this notebook, you will learn:
1. How to enable the peek cache using `bpd.options.compute.enable_peek_cache`.
2. How the peek cache speeds up interactive queries over a large public dataset.
3. How to check execution times between cache hits (local Polars execution) and cache misses (BigQuery execution).


## Setup & Import Libraries


In [ ]:
import time
import bigframes.pandas as bpd

# Configure the BigQuery project to 'bigframes-dev' for this demo
bpd.options.bigquery.project = "bigframes-dev"
bpd.options.bigquery.ordering_mode = 'partial'


# Enable the Local Peek Cache
bpd.options.compute.enable_peek_cache = True
# Set the size of the cached local sample
bpd.options.compute.peek_cache_size = 10000

# Initialize global session
session = bpd.get_global_session()
print(f"Using Google Cloud Project: {session.bqclient.project}")
print(f"Peek cache enabled: {bpd.options.compute.enable_peek_cache}")
print(f"Peek cache size: {bpd.options.compute.peek_cache_size}")


Using Google Cloud Project: bigframes-dev
Peek cache enabled: True
Peek cache size: 10000


## Loading a Large Dataset

We will read from `bigquery-public-data.chicago_taxi_trips.taxi_trips`, which contains over 200 million rows. 

First, let's create a DataFrame representing the table. Since BigQuery DataFrames is lazy by default, this operation is instantaneous and doesn't execute a query yet.


In [9]:
df = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")
print("DataFrame created lazily.")


/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/core/logging/log_adapter.py:183: TimeTravelCacheWarning: Reading cached table from 2026-07-15 22:09:00.959350+00:00 to avoid
incompatibilies with previous reads of this table. To read the latest
version, set `use_cache=False` or close the current session with
Session.close() or bigframes.pandas.close_session().
  return method(*args, **kwargs)


DataFrame created lazily.


## Step 1: The Initial Peek (Cache Miss)

Let's peek at the first 10 rows of our DataFrame. 

Under the hood:
1. Since the peek cache is currently empty, this will result in a **cache miss**.
2. BigQuery DataFrames will issue a query to BigQuery to fetch the first **10,000 rows** (instead of just 10).
3. The 10,000 rows are downloaded and cached locally as a PyArrow table.
4. Finally, the first 10 rows are displayed.

Because this runs a real BigQuery query and downloads data, it will take a few seconds. Let's measure the time:


In [10]:
start_time = time.time()
# Display the preview
print(df.peek(10))
end_time = time.time()
print(f"\nInitial peek execution time: {end_time - start_time:.4f} seconds")


KeyboardInterrupt: 

## Step 2: Interactive Exploration (Cache Hits)

Now that we have the first 10,000 rows cached locally, subsequent interactive transformations and peeks will execute **locally on the cached sample** using Polars.

Let's perform a filter and column projection:
1. Filter trips that cost more than $20 (`fare > 20`).
2. Compute the total cost including tips (`trip_total = fare + tips`).
3. Display the first 10 rows of the result.

This will execute completely locally without querying BigQuery, which means it should take **milliseconds** instead of seconds! Let's measure the time:


In [ ]:
start_time = time.time()

# 1. Filter trips where fare is greater than 20
df_filtered = df[df["fare"] > 20]

# 2. Compute a new column
df_filtered = df_filtered.assign(
    trip_total_calculated = df_filtered["fare"] + df_filtered["tips"]
)

# 3. Select a subset of columns
df_select = df_filtered[["unique_key", "fare", "tips", "trip_total_calculated"]]

# 4. Display/peek the first 10 rows (executed locally on the cache!)
preview_local = df_select.peek(10)
print(preview_local)

end_time = time.time()
print(f"\nSubsequent local peek execution time: {end_time - start_time:.4f} seconds")


                                  unique_key   fare  tips  \
10  86f21e6c68176c64d3e72ed668f111041ded6061   26.5   5.5   
17  6e5d2975e8963793049058f3780b44899f16112f  24.75   0.0   
18  275a2aaf1f291aee60e60818a45aa9fbd53301ac  31.05  10.0   
19  89a54df54be3bbb16f2697aacf9d3f744b06f8d1  31.85   5.0   
21  db5ec3c5a3cb5ad276455fbddd34959cbcc9c315  35.65   0.0   
35  1d793d5961ac65b9328104e5e2ddb728471f25c9  50.05   0.0   
39  a3acd7a40f8293b9779d6c1164e49d6ddc7d0c31   29.0   0.0   
40  fa4f77a18efc0768330b26d3fa202952c97c400b  32.25   5.0   
41  b7f09dbb2ad4cf349acd7971ab0aae4202588efb  20.75   0.0   
43  20b79a0c33129be5b98d1a525f3dace6576060a7   27.5   0.0   

    trip_total_calculated  
10                   32.0  
17                  24.75  
18                  41.05  
19                  36.85  
21                  35.65  
35                  50.05  
39                   29.0  
40                  37.25  
41                  20.75  
43                   27.5  

[10 rows x 4 column

## Step 3: Full Execution (Bypassing the Cache)

The local peek cache is only used when you are *peeking* at a subset of rows (e.g. using `head()`). 

If you perform an operation that requires processing the entire dataset (such as `.to_pandas()`, `.to_gbq()`, or computing the global `.mean()`), BigQuery DataFrames automatically falls back to full BigQuery execution to ensure the results are computed correctly over the entire 200M+ rows.

Let's look at the execution flow of computing the average fare over the entire dataset:


In [ ]:
start_time = time.time()

# Compute the mean fare over the entire dataset (200M+ rows)
# This will execute on BigQuery (bypassing the local cache to ensure correctness)
mean_fare = df["fare"].mean()
print(f"Average fare: ${mean_fare:.2f}")

end_time = time.time()
print(f"Full BigQuery query execution time: {end_time - start_time:.4f} seconds")


Average fare: $13.82
Full BigQuery query execution time: 1.3268 seconds


## Step 4: Aggregation and Complete Cache Exploration

A particularly powerful workflow with the Local Peek Cache is **aggregation followed by rapid exploration**.

If you run an aggregation (like `groupby.agg`) that reduces a massive dataset to a small number of rows (e.g., less than 10,000 rows):
1. Calling a full execution method like `.to_pandas()` or displaying the aggregated DataFrame will query BigQuery once to compute the aggregated data.
2. Because the total number of rows in the aggregated result is smaller than the `peek_cache_size`, the entire result fits in the cache.
3. BigQuery DataFrames caches the result and marks the cache entry as **complete** (`is_complete=True`).
4. **Any subsequent operation** on this aggregated DataFrame (filtering, sorting, column math, etc.) will run **100% locally** using Polars, even if you request full execution (like `.to_pandas()`), without ever querying BigQuery again!

Let's start by calculating the average fare, average tips, and total trip count grouped by payment type over the 200M+ taxi trips table.


In [ ]:
start_time = time.time()

# 1. Group by payment_type and compute stats
df_agg = df.groupby("payment_type").agg(
    fare_mean=("fare", "mean"),
    trip_count=("fare", "count"),
    tips_mean=("tips", "mean")
).reset_index()

# 2. Perform a full download to pandas (initial compilation and BQ query)
print("Running BigQuery aggregation query...")
df_agg_pd = df_agg.to_pandas()
print(df_agg_pd)

end_time = time.time()
print(f"\nAggregation query and download execution time: {end_time - start_time:.4f} seconds")


### Downstream Local Analysis (Cache Hit)

Since the output of the aggregation is small and has been fully downloaded, it is stored in the cache as a **complete** entry. 

Let's perform further exploration on this aggregated data:
1. Filter out payment types with less than 1,000 trips.
2. Calculate the tip percentage (`tip_percentage = (tips_mean / fare_mean) * 100`).
3. Sort by `tip_percentage` descending.
4. Download and display the full table using `.to_pandas()`.

This entire block runs on the local engine (Polars) over the cached Arrow table in **milliseconds**, with zero BigQuery charges!


In [ ]:
start_time = time.time()

# 1. Filter out rare payment types (less than 1,000 trips)
df_common = df_agg[df_agg["trip_count"] >= 1000]

# 2. Compute the tip percentage
df_common = df_common.assign(
    tip_percentage = (df_common["tips_mean"] / df_common["fare_mean"]) * 100
)

# 3. Sort by tip percentage descending
df_sorted = df_common.sort_values(by="tip_percentage", ascending=False)

# 4. Perform a full download to pandas (runs 100% locally on the complete cache!)
result_pd = df_sorted.to_pandas()
print(result_pd)

end_time = time.time()
print(f"\nDownstream local exploration execution time: {end_time - start_time:.4f} seconds")
